# Tutorial: Temporal Profiles, Ramps, and Scenario Authoring

Detailed step-by-step workflow notebook for this SD-dMFA repository.


## Audience, Prerequisites, Outcomes

**Audience**
- Scenario authors defining temporal controls and ramp workflows.

**Prerequisites**
- Python 3.11+ environment for this repo.
- `pip install -e ".[dev]"` completed.
- Notebook executed from repository root or a subfolder.

**Outcomes**
- Understand scalar/year-gate/timeseries/exogenous-ramp forms.
- Inspect and validate ramp profile CSV schema.
- Expand profile CSVs into full-horizon payloads for authoring.


## Outline

1. Recap temporal forms and where used
2. Inspect ramp profile CSV examples
3. Resolve/expand profile payloads
4. Author a new ramp profile template
5. Validate no pre-reporting modifications behavior


In [ ]:
from __future__ import annotations

import json
import os
import subprocess
from pathlib import Path
from textwrap import dedent

import pandas as pd
import matplotlib.pyplot as plt

try:
    from crm_model.common.io import load_run_config
except Exception:
    load_run_config = None

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 240)


In [ ]:
DRY_RUN = False
RUN_HEAVY = False
RUN_PLOTS = False
RUN_CALIBRATION = False
RUN_AUDIT = False

CONFIG = "configs/runs/mvp.yml"
EXAMPLE_VARIANT = "baseline"


In [ ]:
def find_repo_root(start: Path | None = None) -> Path:
    p = (start or Path.cwd()).resolve()
    for cand in [p, *p.parents]:
        if (cand / "configs").exists() and (cand / "src").exists():
            return cand
    raise RuntimeError("Could not locate repo root from current working directory.")


def sh(cmd: str, *, cwd: Path, check: bool = True) -> subprocess.CompletedProcess | None:
    print(f"$ {cmd}")
    if DRY_RUN:
        return None
    cp = subprocess.run(cmd, cwd=str(cwd), shell=True, text=True, capture_output=True)
    if cp.stdout.strip():
        print(cp.stdout)
    if cp.stderr.strip():
        print(cp.stderr)
    if check and cp.returncode != 0:
        raise RuntimeError(f"Command failed ({cp.returncode}): {cmd}")
    return cp


def latest_dir(base: Path) -> Path | None:
    if not base.exists():
        return None
    cands = [p for p in base.iterdir() if p.is_dir() and p.name != "_archive"]
    return sorted(cands)[-1] if cands else None


def load_csv(path: Path) -> pd.DataFrame:
    if not path.exists():
        print(f"Missing: {path}")
        return pd.DataFrame()
    return pd.read_csv(path)


REPO = find_repo_root()
CONFIG_PATH = (REPO / CONFIG).resolve()
CONFIG_STEM = CONFIG_PATH.stem
print("Repo:", REPO)
print("Config:", CONFIG_PATH)


## Step 1: Temporal form recap


Supported forms for temporal-capable keys:

- Scalar: `key: 0.26`
- Year-gate: `key: {start_year: 2025, value: 0.30, before: 0.26}`
- Full timeseries: `key: [..]`
- Exogenous ramp ref: `key: {exogenous_ramp: data/ramp_profiles/...csv}`


## Step 2: Inspect ramp profile CSV inventory


In [ ]:
profiles = sorted((REPO / "data" / "ramp_profiles").glob("**/*.csv"))
print("profile csv count:", len(profiles))
for p in profiles:
    print(" -", p.relative_to(REPO))


## Step 3: Inspect one profile schema and rows


In [ ]:
p = REPO / "data" / "ramp_profiles" / "r_strategies" / "r36_profiles.csv"
if p.exists():
    df = pd.read_csv(p)
    print(df.columns.tolist())
    display(df.head(20))


## Step 4: Expand profiles to full-horizon payload


In [ ]:
cmd = (
    f"python scripts/scenarios/build_reporting_timeseries_profiles.py --config {CONFIG} "
    "--profile data/ramp_profiles/r_strategies/r36_profiles.csv"
)
print(cmd)
if RUN_HEAVY:
    _ = sh(cmd, cwd=REPO)


## Step 5: Create a new ramp profile template row set


In [ ]:
template = pd.DataFrame([
    {"variant": "my_variant", "block": "strategy", "key": "recycling_rate", "year": 2025, "value": 0.62, "material": "", "region": "", "before": ""},
    {"variant": "my_variant", "block": "strategy", "key": "recycling_rate", "year": 2035, "value": 0.70, "material": "", "region": "", "before": ""},
    {"variant": "my_variant", "block": "strategy", "key": "recycling_rate", "year": 2050, "value": 0.76, "material": "", "region": "", "before": ""},
    {"variant": "my_variant", "block": "strategy", "key": "recycling_rate", "year": 2100, "value": 0.76, "material": "", "region": "", "before": ""},
])
out_csv = REPO / "data" / "ramp_profiles" / "templates" / "my_variant_profile_template.csv"
out_csv.parent.mkdir(parents=True, exist_ok=True)
template.to_csv(out_csv, index=False)
print("Wrote", out_csv)
display(template)


## Step 6: Pre-reporting guard check example


In [ ]:
from crm_model.cli import _enforce_reporting_phase_for_variant_slice

cfg = load_run_config(CONFIG_PATH)
years = list(range(cfg.time.start_year, cfg.time.end_year + 1))
report_start = int(cfg.time.report_start_year)

sample = {
    "sd_parameters": {},
    "mfa_parameters": {},
    "strategy": {"recycling_rate": {"start_year": 2015, "value": 0.8}},
    "transition_policy": {},
    "demand_transformation": {},
    "shocks": {},
}

fixed = _enforce_reporting_phase_for_variant_slice(
    variant_slice=sample,
    years=years,
    report_start_year=report_start,
    sd_base={},
    mfa_base={},
    strategy_base={"recycling_rate": 0.57},
    transition_policy_base={},
    demand_transformation_base={},
    shocks_base={},
)
print(fixed["strategy"]["recycling_rate"])


## Pitfalls

- Ramps that omit region/material scope may apply globally by precedence.
- Always verify profile collision behavior when run-level profile overlays are enabled.


## Exercises

1. Repeat this workflow with `CONFIG=configs/runs/r-strategies.yml`.
2. Record one thing that changed and why.
3. Add one guardrail/check specific to your team workflow.


In [ ]:
# Exercise answer scaffold
pass
